XML Parsing

In [71]:
import xml.etree.ElementTree as ET
from datetime import datetime
import numpy as np

Load XML

In [72]:
tree = ET.parse("Instance8.xml")
root = tree.getroot()

Read Planning Period (p)

In [73]:
start = datetime.fromisoformat(root.find("StartDate").text)
end = datetime.fromisoformat(root.find("EndDate").text)
P = (end - start).days + 1
print("Days:", P)

Days: 28


Read shift typed

In [74]:
shift_types = []
shift_dict = {}

for i, sh in enumerate(root.find("ShiftTypes")):
    shift_id = sh.attrib["ID"]   # e.g. "D"
    shift_types.append(shift_id)
    shift_dict[shift_id] = i

NUM_SHIFTS = len(shift_types)
print("Shifts:", shift_types)
print("Shift", shift_dict)

Shifts: ['E', 'D', 'L', 'N']
Shift {'E': 0, 'D': 1, 'L': 2, 'N': 3}


Read Employees

In [75]:
employees = []
employee_contracts = {}

for i, emp in enumerate(root.find("Employees")):
    emp_id = emp.attrib["ID"]
    key=i
    employees.append(emp_id)
    employee_contracts[emp_id] = [
        c.text for c in emp.findall("ContractID")
    ]

S = len(employees)

#print("Nurses:", employees)



emp_dict = {}
for l, ip in enumerate(employees):
    # print(f"{l}. Contract of {ip} is {employee_contracts[ip]}")

    emp_dict[ip] = l

print("Nurses with Indices\n",emp_dict)

print("Contracts for each Nurse with Key\n",employee_contracts)



Nurses with Indices
 {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25, 'AA': 26, 'AB': 27, 'AC': 28, 'AD': 29}
Contracts for each Nurse with Key
 {'A': ['All', 'A'], 'B': ['All', 'B'], 'C': ['All', 'C'], 'D': ['All', 'D'], 'E': ['All', 'E'], 'F': ['All', 'F'], 'G': ['All', 'G'], 'H': ['All', 'H'], 'I': ['All', 'I'], 'J': ['All', 'J'], 'K': ['All', 'K'], 'L': ['All', 'L'], 'M': ['All', 'M'], 'N': ['All', 'N'], 'O': ['All', 'O'], 'P': ['All', 'P'], 'Q': ['All', 'Q'], 'R': ['All', 'R'], 'S': ['All', 'S'], 'T': ['All', 'T'], 'U': ['All', 'U'], 'V': ['All', 'V'], 'W': ['All', 'W'], 'X': ['All', 'X'], 'Y': ['All', 'Y'], 'Z': ['All', 'Z'], 'AA': ['All', 'AA'], 'AB': ['All', 'AB'], 'AC': ['All', 'AC'], 'AD': ['All', 'AD']}


Build demand matrix from <CoverRequirements> 

We build:
DEMAND[day][shift] = required nurses

In [76]:
DEMAND = np.zeros((P, NUM_SHIFTS), dtype=int)

for cover_day in root.find("CoverRequirements"):
    day = int(cover_day.find("Day").text)
    cover = cover_day.find("Cover")
    sh = cover.find("Shift").text
    min_required = int(cover.find("Min").text)

    DEMAND[day, shift_dict[sh]] = min_required

print("DEMAND MATRIX FOR EACH DAY")

for i,_ in enumerate(DEMAND):
    print(f"{i}: {DEMAND[i]}")


DEMAND MATRIX FOR EACH DAY
0: [5 0 0 0]
1: [6 0 0 0]
2: [5 0 0 0]
3: [5 0 0 0]
4: [5 0 0 0]
5: [5 0 0 0]
6: [5 0 0 0]
7: [5 0 0 0]
8: [4 0 0 0]
9: [5 0 0 0]
10: [6 0 0 0]
11: [5 0 0 0]
12: [5 0 0 0]
13: [5 0 0 0]
14: [5 0 0 0]
15: [4 0 0 0]
16: [3 0 0 0]
17: [3 0 0 0]
18: [6 0 0 0]
19: [4 0 0 0]
20: [5 0 0 0]
21: [5 0 0 0]
22: [5 0 0 0]
23: [3 0 0 0]
24: [3 0 0 0]
25: [3 0 0 0]
26: [5 0 0 0]
27: [3 0 0 0]


Fixed Assignments

In [77]:
fixed_assignments = []

for emp in root.find("FixedAssignments"):
    emp_id = emp.find("EmployeeID").text
    assign = emp.find("Assign")
    shift = assign.find("Shift").text
    day = int(assign.find("Day").text)

    fixed_assignments.append((day, shift, emp_id))
w=len(fixed_assignments)

fixed_assignments.sort()
for i in range(w):
    print(fixed_assignments[i])
#print(f"{fixed_assignments[i] for i in range(w)}")
print("Total fixed assignemnt",w)

(0, '-', 'M')
(1, '-', 'A')
(1, '-', 'W')
(2, '-', 'A')
(2, '-', 'E')
(3, '-', 'J')
(3, '-', 'Q')
(3, '-', 'X')
(4, '-', 'B')
(4, '-', 'R')
(4, '-', 'S')
(4, '-', 'X')
(5, '-', 'AB')
(5, '-', 'Y')
(6, '-', 'AB')
(7, '-', 'P')
(7, '-', 'R')
(7, '-', 'T')
(8, '-', 'N')
(8, '-', 'P')
(9, '-', 'B')
(9, '-', 'L')
(10, '-', 'J')
(10, '-', 'V')
(11, '-', 'H')
(12, '-', 'H')
(12, '-', 'M')
(12, '-', 'Q')
(13, '-', 'G')
(13, '-', 'I')
(14, '-', 'E')
(14, '-', 'G')
(14, '-', 'I')
(14, '-', 'K')
(14, '-', 'L')
(14, '-', 'O')
(15, '-', 'AC')
(15, '-', 'AD')
(15, '-', 'K')
(16, '-', 'AC')
(16, '-', 'AD')
(16, '-', 'C')
(17, '-', 'C')
(17, '-', 'Z')
(18, '-', 'V')
(18, '-', 'Z')
(19, '-', 'W')
(21, '-', 'U')
(22, '-', 'D')
(22, '-', 'U')
(23, '-', 'AA')
(24, '-', 'AA')
(24, '-', 'O')
(24, '-', 'S')
(25, '-', 'Y')
(26, '-', 'D')
(26, '-', 'F')
(27, '-', 'F')
(27, '-', 'N')
(27, '-', 'T')
Total fixed assignemnt 60


Genetic Algorithm

Chromosome Representation

# schedule[nurse][day][shift] ∈ {0,1}
def random_schedule():
    sched = np.zeros((S, P, NUM_SHIFTS), dtype=int)

    for n in range(S):
        for d in range(P):
            if np.random.rand() < 0.6:
                sh = np.random.randint(NUM_SHIFTS)
                sched[n, d, sh] = 1
    return sched

Fitness Function

In [78]:
def random_schedule():
    sched = np.zeros((S, P, NUM_SHIFTS), dtype=int)

    for i in range(w):

        day, shift, nurse_id = fixed_assignments[i]
        nurse = emp_dict[nurse_id]

        if shift=='-':
            shift = np.random.randint(NUM_SHIFTS)
            sched[nurse, day, shift] = 1
        else:
            shift = shift_dict[shift]
            sched[nurse, day, shift] = 1
                
    for n in range(S):
        for d in range(P):
            if np.random.rand() < 0.6:
                sh = np.random.randint(NUM_SHIFTS)
                sched[n, d, sh] = 1
    return sched

In [ ]:

HARD_PENALTY = 20
HIGH_PENALTY = 8
FIVE_W = 2
TWO_W = 1


def fitness(schedule):
    penalty = 0

    # ---- Hard: one shift per nurse per day
    penalty += np.sum(schedule.sum(axis=2) > 1) * HARD_PENALTY

    # ---- Hard: cover demand
    for d in range(P):
        for sh in range(NUM_SHIFTS):
            assigned = schedule[:, d, sh].sum()
            if assigned < DEMAND[d, sh]:
                penalty += (DEMAND[d, sh] - assigned) * HARD_PENALTY
    
    # ---- Hard: max 2 consecutive night shift AND OFF-day
    if 'N' in shift_types:
        for n in range(S):
            consec = 0
            for d in range(P):
                if schedule[n, d].sum() == 0:
                    consec=0
                    continue
                for sh in range(NUM_SHIFTS):
                    if schedule[n,d,sh] == 1:
                        temp=shift_types.index('N')
                        if sh == temp:
                            consec+=1
                        else:
                            consec=0
                if consec>2: #Max 2 consecutive night shift
                    penalty += HARD_PENALTY 
                if consec==2:
                    if d+1 in range(P):
                        if (schedule[n,d+1].sum() > 0): #OFF-day after night shift
                            penalty += HARD_PENALTY

    # ---- High: fixed assignments
    for day, sh, emp_id in fixed_assignments:
        n = employees.index(emp_id)
        if sh == "-":
            if schedule[n, day].sum() > 0:
                continue
            else:
               penalty += High_PENALTY 
        else:
            s_idx = shift_dict[sh]
            if schedule[n, day, s_idx] == 0:
                penalty += HIGH_PENALTY

    # ---- Soft: max 5 consecutive working days
    for n in range(S):
        consec = 0
        for d in range(P):
            if schedule[n, d].sum() > 0:
                consec += 1
                if consec > 5:
                    penalty += FIVE_W
            else:
                consec = 0

    # ---- Soft: min 2 consecutive working days
    for n in range(S):
        consec = 0
        for d in range(P):
            if schedule[n, d].sum() > 0:
                consec += 1
                if consec < 2:
                    penalty += TWO_W
            else:
                consec = 0
                
    return -penalty

GA Loop

In [80]:
POP = 100
GENS = 500
MUT_RATE = 0.01
C_over_RATE = 0.5

population = [random_schedule() for _ in range(POP)]

#print("random population", population)

for gen in range(GENS):
    scores = [fitness(ind) for ind in population]

    new_pop = []
    elite = population[np.argmax(scores)]
    new_pop.append(elite.copy())

    while len(new_pop) < POP:
        indices=np.random.choice(len(population),10)
        indices2=np.random.choice(len(population),10)
        p1 = population[indices[np.argmax([scores[i] for i in indices])]] #tournament
        p2 = population[indices2[np.argmax(scores[i] for i in indices2)]] #tournament
        child = p1.copy()
        mask = np.random.rand(S, P, NUM_SHIFTS) < C_over_RATE 
        child[mask] = p2[mask]
        
        #Mutation
        if np.random.rand() < MUT_RATE:  # 20% chance to mutate this child
            i = np.random.randint(S)
            j = np.random.randint(P)
            k = np.random.randint(NUM_SHIFTS)
            child[i, j, k] = np.random.randint(0, 2)
        

        new_pop.append(child)

    population = new_pop[:POP]

#print("selected some", population)

Output

In [81]:
best = population[np.argmax([fitness(ind) for ind in population])]

fit_value=fitness(best)

print("Fitness Value = ", fit_value)

print("\nFinal Schedule:")
for d in range(P):
    for sh, name in enumerate(shift_types):
        row = [
            "Y" if best[n, d, sh] else "N"
            for n in range(S)
        ]
        print(f"Day {d}, Shift {name}: {row}")

Fitness Value =  -409

Final Schedule:
Day 0, Shift E: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'Y', 'Y', 'Y', 'N', 'Y', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N']
Day 0, Shift D: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'N', 'Y']
Day 0, Shift L: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N']
Day 0, Shift N: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'Y', 'Y', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N']
Day 1, Shift E: ['Y', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'Y', 'Y', 'Y', 'Y', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N', 'Y', 'N', 'N', 'N', 'N', 'N']
Day 1, Shift D: ['N', 'N', 'Y', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'Y', 'N', 'N',